In [7]:
# Regenerating with N=500 to be more conservative
import os, json, random, uuid, datetime, pathlib, sys, time
OUT_DIR = "../data/synthetic_advanced"
pathlib.Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

In [2]:
# Minimal setup using previous definitions (redefine compactly)
INTENTS = {
    "CARD_REPLACEMENT": {"templates":["I need a replacement card sent to my new address.","My card is lost/stolen; please send a new card.","Can you replace my damaged card?"]},
    "PAYMENT_DUE_DATE": {"templates":["What's my payment due date?","When do I need to pay my credit card bill?","Tell me the due date for my payment."]},
    "TRANSACTION_DISPUTE": {"templates":["I see a charge I don't recognize.","There is an unauthorized transaction on my account.","I want to dispute a recent transaction."]},
    "ADDRESS_UPDATE": {"templates":["I moved to a new address; please update my mailing address.","Change my shipping address to 123 New St.","Update address for card delivery."]},
    "STATEMENT_REQUEST": {"templates":["I did not receive my statement.","Send me my monthly statement.","I need a copy of last month's statement."]},
    "GENERAL_QUERY": {"templates":["I have a question about my account.","How do I change my contact preferences?","Tell me more about my rewards."]}
}
FLOWS = {
    "CARD_REPLACEMENT":[["USER_MSG","AGENT_ASK_AUTH","AUTH_CALL","AUTH_RESPONSE","AGENT_TOOL_CALL","TOOL_RESPONSE","AGENT_CLOSE"],["USER_MSG","AGENT_ASK_AUTH","AUTH_CALL","AUTH_FAIL","AGENT_ASK_CLARIFY","AUTH_CALL","AUTH_RESPONSE","AGENT_TOOL_CALL","TOOL_RESPONSE","AGENT_CLOSE"]],
    "PAYMENT_DUE_DATE":[["USER_MSG","AGENT_SIMPLE_RESPONSE","AGENT_CLOSE"],["USER_MSG","AGENT_CHECK_ACCOUNT","TOOL_CALL","TOOL_RESPONSE","AGENT_CLOSE"]],
    "TRANSACTION_DISPUTE":[["USER_MSG","AGENT_ASK_DETAILS","AGENT_TOOL_CALL","TOOL_RESPONSE","AGENT_ESCALATE","AGENT_CLOSE"],["USER_MSG","AGENT_ASK_DETAILS","AGENT_TOOL_CALL","TOOL_FAIL","AGENT_ESCALATE","AGENT_CLOSE"]],
    "ADDRESS_UPDATE":[["USER_MSG","AGENT_ASK_AUTH","AUTH_CALL","AUTH_RESPONSE","AGENT_UPDATE_ADDRESS","AGENT_CLOSE"]],
    "STATEMENT_REQUEST":[["USER_MSG","AGENT_TOOL_CALL","TOOL_RESPONSE","AGENT_CLOSE"]],
    "GENERAL_QUERY":[["USER_MSG","AGENT_SIMPLE_RESPONSE","AGENT_CLOSE"]]
}
TOOL_NAMES = ["Authentication","Transaction_Lookup","SMS","Send_Replacement_Card","Address_Update"]
AGENT_UTTERANCES = {
    "AGENT_ASK_AUTH":["Can you confirm your account token or last 4 digits of SSN?","I need to verify your identity, please provide the account token."],
    "AGENT_SIMPLE_RESPONSE":["Sure, I can help with that.","I can check that for you."],
    "AGENT_CHECK_ACCOUNT":["I'll check your account details.","Let me look that up."],
    "AGENT_ASK_DETAILS":["Can you provide the transaction date and amount?","Please share more details about the transaction."],
    "AGENT_UPDATE_ADDRESS":["I have updated your address.","Your address has been changed and a new card will be sent."],
    "AGENT_ESCALATE":["I'll escalate this to the investigations team.","This will be escalated for further review."],
    "AGENT_CLOSE":["Is there anything else I can help you with?","Thank you for contacting us. Have a good day!"],
    "AGENT_ASK_CLARIFY":["Can you re-confirm the phone number?","I didn't catch that, could you repeat?"]
}
SENTIMENT_TRAJECTORIES = ["neutral_to_positive","neutral_to_negative","negative_to_positive","stable_neutral","fluctuating"]

In [3]:
def utc_now_iso():
    return datetime.datetime.utcnow().isoformat() + "Z"
def choose_intent():
    return random.choice(list(INTENTS.keys()))
def sample_user_utterance(intent):
    return random.choice(INTENTS[intent]["templates"])
def make_event(event_name, role, text, event_data=None):
    return {"event_id": str(uuid.uuid4()), "event_name": event_name, "event_timestamp": utc_now_iso(), "participant": {"participant_id": f"{role}_{random.randint(1000,9999)}", "role": role}, "text": text, "event_data": event_data or {}}
def make_event_toolcall(event_name, role, text, tool_name, arguments, tool_call_id):
    return make_event(event_name, role, text, {"tool_name": tool_name, "arguments": arguments, "tool_call_id": tool_call_id})
def make_event_toolresponse(event_name, role, text, response_obj):
    return make_event(event_name, role, text, response_obj)

def simulate_flow(intent, flow_template):
    events = []
    events.append(make_event("USER_MESSAGE_RECEIVED","user", sample_user_utterance(intent)))
    for state in flow_template[1:]:
        if state.startswith("AGENT"):
            events.append(make_event("AGENT_MESSAGE_SENT","agent", random.choice(AGENT_UTTERANCES.get(state, ["Okay."]))))
        elif state=="AUTH_CALL":
            tcid=str(uuid.uuid4()); events.append(make_event_toolcall("AGENT_TOOL_CALLED","agent","Calling Authentication","Authentication",{"method":"last4SSN","value":"XXXX"},tcid))
        elif state=="AUTH_RESPONSE":
            last_tcid=events[-1]["event_data"].get("tool_call_id"); resp={"tool_name":"Authentication","tool_call_id":last_tcid,"status":"success","latency_ms": random.randint(50,500)}; events.append(make_event_toolresponse("TOOL_RESPONSE_RECEIVED","system","Authentication success",resp))
        elif state=="AUTH_FAIL":
            last_tcid=events[-1]["event_data"].get("tool_call_id"); resp={"tool_name":"Authentication","tool_call_id":last_tcid,"status":"fail","error_code":"AUTH_FAIL","latency_ms": random.randint(50,500)}; events.append(make_event_toolresponse("TOOL_RESPONSE_RECEIVED","system","Authentication failed",resp))
        elif state in ("AGENT_TOOL_CALL","TOOL_CALL"):
            tool=random.choice(TOOL_NAMES); tcid=str(uuid.uuid4()); events.append(make_event_toolcall("AGENT_TOOL_CALLED","agent",f"Calling {tool}",tool,{"query":"details"},tcid))
        elif state=="TOOL_RESPONSE":
            last=events[-1]; tcid=last["event_data"].get("tool_call_id"); 
            if random.random()<0.85: resp={"tool_name": last["event_data"].get("tool_name"), "tool_call_id": tcid, "status":"success", "response":{"info":"ok"}, "latency_ms": random.randint(50,1500)}
            else: resp={"tool_name": last["event_data"].get("tool_name"), "tool_call_id": tcid, "status":"fail", "error_code":"INVALID_PHONE", "latency_ms": random.randint(50,1500)}
            events.append(make_event_toolresponse("TOOL_RESPONSE_RECEIVED","system","Tool responded",resp))
        elif state=="TOOL_FAIL":
            last=events[-1]; tcid=last["event_data"].get("tool_call_id"); resp={"tool_name": last["event_data"].get("tool_name"), "tool_call_id": tcid, "status":"fail", "error_code":"TIMEOUT", "latency_ms": random.randint(1500,5000)}; events.append(make_event_toolresponse("TOOL_RESPONSE_RECEIVED","system","Tool failed",resp))
        elif state=="AGENT_UPDATE_ADDRESS":
            events.append(make_event("AGENT_MESSAGE_SENT","agent","Your address has been updated."))
        elif state=="AGENT_ASK_CLARIFY":
            events.append(make_event("AGENT_MESSAGE_SENT","agent", random.choice(AGENT_UTTERANCES["AGENT_ASK_CLARIFY"])))
        elif state=="AGENT_CLOSE":
            events.append(make_event("AGENT_MESSAGE_SENT","agent", random.choice(AGENT_UTTERANCES["AGENT_CLOSE"])))
        elif state=="AGENT_ESCALATE":
            events.append(make_event("AGENT_MESSAGE_SENT","agent", random.choice(AGENT_UTTERANCES["AGENT_ESCALATE"])))
        else:
            events.append(make_event("AGENT_MESSAGE_SENT","agent","Okay."))
    return events

def simulate_sentiment_trajectory(num_turns, trajectory):
    if num_turns<=1: return [0.0]
    scores=[]
    for i in range(num_turns):
        if trajectory=="neutral_to_positive": s=min(1.0, -0.1 + 0.03*i + random.uniform(-0.05,0.05))
        elif trajectory=="neutral_to_negative": s=max(-1.0, 0.1 - 0.04*i + random.uniform(-0.05,0.05))
        elif trajectory=="negative_to_positive": s=-0.6 + (1.6)*(i/(num_turns-1)) + random.uniform(-0.05,0.05)
        elif trajectory=="stable_neutral": s=random.uniform(-0.1,0.1)
        else: s=random.uniform(-0.6,0.6)
        scores.append(max(-1.0,min(1.0,s)))
    return scores

def generate_transcript_with_labels(intent=None):
    if intent is None: intent=choose_intent()
    flow_templates=FLOWS.get(intent,[["USER_MSG","AGENT_SIMPLE_RESPONSE","AGENT_CLOSE"]])
    flow_template=random.choice(flow_templates)
    events=simulate_flow(intent,flow_template)
    traj=random.choice(SENTIMENT_TRAJECTORIES)
    sentiment_scores=simulate_sentiment_trajectory(len(events),traj)
    for i,ev in enumerate(events):
        ev["sentiment_score"]=sentiment_scores[i]
    primary_intent=intent
    authenticated=any(ev["event_name"]=="TOOL_RESPONSE_RECEIVED" and ev["event_data"].get("tool_name")=="Authentication" and ev["event_data"].get("status")=="success" for ev in events)
    tool_failures=[ev for ev in events if ev["event_name"]=="TOOL_RESPONSE_RECEIVED" and ev["event_data"].get("status")=="fail"]
    tool_failure_flag=len(tool_failures)>0
    failure_reasons=[ev["event_data"].get("error_code") for ev in tool_failures if ev["event_data"].get("error_code")]
    user_scores=[ev["sentiment_score"] for ev in events if ev["participant"]["role"]=="user"]
    sentiment_overall=sum(user_scores)/len(user_scores) if user_scores else 0.0
    turn_count=len(events)
    tool_summary=[{"tool_name":ev["event_data"].get("tool_name"),"status":ev["event_data"].get("status"),"error_code":ev["event_data"].get("error_code"),"latency_ms":ev["event_data"].get("latency_ms")} for ev in events if ev["event_name"]=="TOOL_RESPONSE_RECEIVED"]
    transcript={"tenant_id":str(uuid.uuid4()),"conversation_id":f"{uuid.uuid4()}@0.0.0.0","correlation_id":str(uuid.uuid4()),"event_month":datetime.date.today().replace(day=1).isoformat(),"event_timestamp":events[0]["event_timestamp"],"events":events,"schema_version":1,"ingestion_timestamp":utc_now_iso(),"gt_primary_intent":primary_intent,"gt_authenticated":authenticated,"gt_tool_failure":tool_failure_flag,"gt_failure_reasons":failure_reasons,"gt_sentiment_overall":sentiment_overall,"gt_sentiment_trajectory":traj,"gt_turn_count":turn_count,"tool_summary":tool_summary}
    return transcript

In [8]:
N=500
for i in range(N):
    t=generate_transcript_with_labels()
    fname=os.path.join(OUT_DIR,f"transcript_{i:04d}.json")
    with open(fname,"w") as f:
        json.dump(t,f,indent=2)
    if (i+1) % 50 == 0:
        print(f"{i+1} transcripts generated...")
print(f"Generated {N} transcripts in {OUT_DIR}")

50 transcripts generated...
100 transcripts generated...
150 transcripts generated...
200 transcripts generated...
250 transcripts generated...
300 transcripts generated...
350 transcripts generated...
400 transcripts generated...
450 transcripts generated...
500 transcripts generated...
Generated 500 transcripts in ../data/synthetic_advanced
